<a href="https://colab.research.google.com/github/Annie00000/Project/blob/main/1_16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. 從 「Bad 資料的角度出發」，看這些壞掉的資料是否都「縮」在某個機台的特定類別裡。

所有的壞片中，有多少比例是來自這台機台的某個類別？

**概念差別**
- 我之前的（機台勝率）：機台 A 做了 10 片，10 片都壞了（純度高），但可能全製程有 1000 片壞片，它只佔了 1%。

- 你現在要的（壞片集中度）：全製程有 1000 片壞片，其中 900 片都集中在機台 B 的 Tool_#01（集中度極高）。這台機台就是「頭號重災區」。

In [ ]:
# ==========================================
# 更新版：老闆特調評分邏輯 (Bad 集中度導向)
# ==========================================

# 1. 需求 2: 指標任一顯著強化 (L2 Norm)
s_base = np.sqrt((res['O_Mean']**2 + res['O_Risk']**2) / 2)

# 2. 需求 1: Bad 資料在機台類別中的集中度 (老闆修正版)
# 算的是：該機台某類別佔了總壞片的多少比例
ct_focus = pd.crosstab(df_proc[col], df_proc[cluster_col], normalize='columns')
max_focus = ct_focus[bad_label].max() if bad_label in ct_focus.columns else 0

# 結合 T 分數，確保這種集中不是因為樣本太少導致的巧合
s_concentration = max_focus * res['Adj_T']

# 3. 最終合成評分 (權重 0.6 / 0.4)
raw_score = (0.6 * s_base) + (0.4 * s_concentration)

final_data.append({
    'Factor': col,
    'Impact_Score_Raw': raw_score,
    'Mean_Impact': res['O_Mean'],
    'Risk_Impact': res['O_Risk'],
    'Bad_Concentration': max_focus,  # 這是你要的：壞資料集中在該機台的程度
    'Adj_Correlation': res['Adj_T']
})

**改動後的分析效果**
- 情境：總共有 100 片壞片。

- 結果 A：機台 X 的 Tool_01 貢獻了 95 片壞片。那麼 Bad_Concentration 就是 0.95。這台機台會被暴力拉到第一名。

- 結果 B：機台 Y 雖然自己產出的壞片率很高，但它產量很小，100 片壞片中它只貢獻了 2 片。那麼 Bad_Concentration 就是 0.02。它的排名就不會那麼靠前。

## 2.

**引入「有效自由度」懲罰** :

我們目前的 $\omega^2$ 雖然已經有修正，但在「類別數 $k$ 接近樣本數 $n$」時，依然會失真。我們可以對 s_base 增加一個基於類別數量的抑制項。如果類別數超過樣本數的 50%（代表這個因子快變成 Unique ID 了），就大幅扣分。

In [ ]:
# ==========================================
# 更新版：加入過擬合懲罰 (Cardinality Penalty)
# ==========================================

# 1. 取得該因子的類別數 k 與樣本數 n
k_categories = df_proc[col].nunique()
n_total = len(df_proc)

# 2. 計算懲罰係數 (當 k 越大，penalty 越重，值越小)
# 如果 k 佔了總樣本的 50% 以上，這個係數會變得非常小
cardinality_ratio = (k_categories - 1) / (n_total - 1)
complexity_penalty = np.exp(-5 * cardinality_ratio) # 使用指數衰減，對高維度極度敏感

# 3. 修正 s_base
s_base = np.sqrt((res['O_Mean']**2 + res['O_Risk']**2) / 2)
s_base_adj = s_base * complexity_penalty # 套用懲罰

# 4. 最終合成評分
raw_score = (0.6 * s_base_adj) + (0.4 * s_concentration)

## 3.  Final

In [ ]:
import pandas as pd
import numpy as np
import hashlib
from concurrent.futures import ProcessPoolExecutor
from scipy.stats import chi2_contingency

# ==========================================
# 1. 核心指標計算函式
# ==========================================

def _calculate_omega_sq_v2(df, f_col, y_col):
    """計算均值偏移的 Omega-squared (排除 NA 版)"""
    clean_df = df[[f_col, y_col]].dropna()
    n, k = len(clean_df), clean_df[f_col].nunique()
    if k < 2 or n <= k: return 0.0

    y = clean_df[y_col]
    sst = np.sum((y - y.mean())**2)
    if sst <= 0: return 0.0

    ssb = np.sum(y.groupby(clean_df[f_col]).count() * (y.groupby(clean_df[f_col]).mean() - y.mean())**2)
    msw = (sst - ssb) / (n - k)
    return max(0, (ssb - (k - 1) * msw) / (sst + msw))

def _calculate_quantile_risk_omega(df, f_col, y_col, q=0.95):
    """計算分位距風險的 Omega-squared (針對離群點)"""
    clean_df = df[[f_col, y_col]].dropna().copy()
    if len(clean_df) < 5: return 0.0

    q_threshold = clean_df[y_col].quantile(q)
    clean_df['y_risk'] = (clean_df[y_col] - q_threshold).clip(lower=0)

    y_r = clean_df['y_risk']
    n, k = len(y_r), clean_df[f_col].nunique()
    if k < 2 or n <= k: return 0.0

    sst = np.sum((y_r - y_r.mean())**2)
    if sst <= 0: return 0.0

    ssb = np.sum(y_r.groupby(clean_df[f_col]).count() * (y_r.groupby(clean_df[f_col]).mean() - y_r.mean())**2)
    msw = (sst - ssb) / (n - k)
    return max(0, (ssb - (k - 1) * msw) / (sst + msw))

def _calculate_adj_tschuprow_t(df, f_col, c_col):
    """計算 Tschuprow's T 與密度懲罰"""
    clean_df = df[[f_col, c_col]].dropna()
    n = len(clean_df)
    if n < 5: return 0.0

    ct = pd.crosstab(clean_df[f_col], clean_df[c_col])
    r, k = ct.shape
    if r < 2 or k < 2: return 0.0

    chi2 = chi2_contingency(ct, correction=False)[0]
    t_score = np.sqrt((chi2 / n) / np.sqrt((r - 1) * (k - 1)))
    # 內建密度懲罰：當類別數 r 接近樣本數 n 時，分數趨近於 0
    penalty = 1 - (((r - 1) / (n - 1)) ** 0.5)
    return max(0, t_score * penalty)

# ==========================================
# 2. 並行運算封裝
# ==========================================

def _worker_task(args):
    df_small, f_col, y_col, c_col, q = args
    return {
        'O_Mean': _calculate_omega_sq_v2(df_small, f_col, y_col),
        'O_Risk': _calculate_quantile_risk_omega(df_small, f_col, y_col, q),
        'Adj_T': _calculate_adj_tschuprow_t(df_small, f_col, c_col),
        'K_Count': df_small[f_col].nunique(),
        'N_Count': len(df_small)
    }

# ==========================================
# 3. 主分析引擎
# ==========================================

def run_advanced_rca(df, value_col, cluster_col, factor_list, bad_label, q=0.95):
    df_proc = df.copy()

    # 預處理：填補空值與低頻合併
    for col in factor_list:
        df_proc[col] = df_proc[col].fillna('NA').astype(str)
        counts = df_proc[col].value_counts(normalize=True)
        df_proc[col] = df_proc[col].replace(counts[counts < 0.01].index, 'Other_Small')

    # Hashing 加速
    unique_patterns = {}
    col_to_hash = {}
    for col in factor_list:
        h = hashlib.md5(df_proc[col].values.tobytes()).hexdigest()
        col_to_hash[col] = h
        if h not in unique_patterns: unique_patterns[h] = col

    print(f"🚀 分析啟動，處理 {len(unique_patterns)} 種唯一數據模式...")

    tasks = [(df_proc[[unique_patterns[h], value_col, cluster_col]], unique_patterns[h], value_col, cluster_col, q)
             for h in unique_patterns.keys()]

    with ProcessPoolExecutor() as executor:
        results = list(executor.map(_worker_task, tasks))
    cache = dict(zip(unique_patterns.keys(), results))

    final_data = []
    for col in factor_list:
        res = cache[col_to_hash[col]]

        # --- [關鍵更新] 1. 計算複雜度懲罰 (Complexity Penalty) ---
        # 避免像 wafer_id 這種高維度欄位過擬合
        cardinality_ratio = (res['K_Count'] - 1) / (res['N_Count'] - 1)
        # 使用指數衰減：類別佔比越高，懲罰越重 (此處 -5 可依嚴厲程度調整)
        complexity_penalty = np.exp(-5 * cardinality_ratio)

        # 需求 2: 指標任一顯著強化 (標準化 L2 Norm) + 套用懲罰
        s_base = np.sqrt((res['O_Mean']**2 + res['O_Risk']**2) / 2)
        s_base_adj = s_base * complexity_penalty

        # --- [關鍵更新] 2. Bad 集中度 (Concentration) ---
        # 算的是：所有的壞片中，有多少比例集中在該機台的單一類別內
        ct_focus = pd.crosstab(df_proc[col], df_proc[cluster_col], normalize='columns')
        max_focus = ct_focus[bad_label].max() if bad_label in ct_focus.columns else 0

        # 集中度同樣受 Adj_T 監督，避免隨機巧合
        s_concentration = max_focus * res['Adj_T']

        # 3. 合成評分 (0.6 * 基礎影響力 + 0.4 * 壞片集中度)
        raw_score = (0.6 * s_base_adj) + (0.4 * s_concentration)

        final_data.append({
            'Factor': col,
            'Impact_Score_Raw': raw_score,
            'Mean_Impact': res['O_Mean'],
            'Risk_Impact': res['O_Risk'],
            'Bad_Concentration': max_focus,
            'Complexity_Penalty': complexity_penalty, # 方便除錯
            'Adj_T': res['Adj_T']
        })

    report = pd.DataFrame(final_data)
    max_raw = report['Impact_Score_Raw'].max()
    report['Final_Rank_Score'] = (report['Impact_Score_Raw'] / max_raw * 100).round(1) if max_raw > 0 else 0

    return report.sort_values('Final_Rank_Score', ascending=False).reset_index(drop=True)

### 🛠️ 此版本的四大進化：

1. **自動過濾 `wafer_id**`：
透過 `complexity_penalty`，當一個欄位的類別數（K）太多時，`np.exp(-5 * K/N)` 會變成一個極小的數字（如 0.01）。即便該欄位的 `Mean_Impact` 是 1.0，乘上懲罰後也會變得很低，成功解決過擬合。
2. **修正「集中度」邏輯**：
現在 `Bad_Concentration` 反映的是：**「如果這一批有 100 個壞點，這台機台佔了其中幾個？」** 這能精準回應老闆對於「重災區」的定義。
3. **標準化  範數**：
使用了 `/ 2` 的平方根，確保  $S_{base}$ 嚴格落在0-1 ，讓權重分配（0.6 與 0.4）具有物理意義。
4. **透明的懲罰資訊**：
輸出結果中包含 `Complexity_Penalty` 欄位。如果一個你認為很重要的因子排名很低，你可以檢查這一欄，看看是不是因為它的類別分得太細被系統「沒收」分數了。

這套邏輯現在非常強健，既能抓到「偶發噴訊」，又能防範「維度陷阱」。你可以放心拿去跑那 2000 筆資料了！